In [ ]:
# === CABECALHO ACADEMICO (gerado por scripts/apply_header.py) ===
# Universidade Presbiteriana Mackenzie
# Disciplina: Inteligência Artificial
# Projeto: Previsão de desempenho de jogadores da NBA (pontos por jogo)
#
# INTEGRANTES DO GRUPO:
#   - Rodrigo Lucas Mascarenhas Leite Oliveira - TIA 10427925
#   - André Ihsan Ward - TIA 10425684
#
# ARQUIVO: notebooks/03_classificacao_e_calibracao.ipynb
# SINTESE: Classificação contra o patamar recente e calibração.
#
# HISTORICO DE ALTERACOES:
#   data       | autor      | descricao
#   -----------|------------|-------------------------------------------
#   2026-11-15 | Rodrigo    | Versão inicial.
# === FIM DO CABECALHO ===


# 03 - Classificação contra o patamar recente e calibração

Alvo: `y_over` = 1 se o jogador marcou **mais** que a mediana dos seus 10 jogos anteriores (`line_synthetic`). A pergunta é se o modelo produz **probabilidades calibradas**, não se acerta muito: como a linha acompanha o próprio jogador, o teto de acurácia é baixo.

**Correção em relação à N1:** a mediana de 10 inteiros pode ser inteira (ex.: 12,0), e o arredondamento para ,5 não muda isso. Nesses casos pode haver **empate** (`y_pts == linha`), rotulado como "não superou". Reportamos tudo com e sem empates.

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
from IPython.display import Image, display
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
pd.set_option("display.max_columns", 30)
TV = json.loads(Path("results/metrics/treino_validacao.json").read_text(encoding="utf-8"))
TF = json.loads(Path("results/metrics/test_final.json").read_text(encoding="utf-8"))

In [ ]:
from src.modeling import load_dataset, add_clf_features, clf_feature_set, CLF_EXTRA_FEATURES
df = add_clf_features(load_dataset())
emp = df["y_pts"] == df["line_synthetic"]
print(f"linha inteira em {df['linha_inteira'].mean():.1%} das linhas; empates: {emp.sum():,} ({emp.mean():.1%}), todos com y_over = {df.loc[emp, 'y_over'].unique()}")
df.groupby("linha_inteira")["y_over"].mean().rename("taxa de superação").to_frame()

## 1. Por que o classificador precisa ver a linha

Sozinhas, as médias recentes quase não informam se o jogador vai superar a **própria** mediana (a linha sobe e desce junto com elas). Boa parte do sinal está na **diferença** entre a média recente e a linha. Por isso o modelo recebe a linha e as diferenças: `diff_ewm_linha, diff_ma5_linha, diff_media_temporada_linha, z_ewm_linha, linha_inteira`.

In [ ]:
from sklearn.metrics import roc_auc_score
tr = df[df["split"] == "train"]
pd.Series({c: roc_auc_score(tr["y_over"], tr[c]) for c in ["pts_ewm_prev", "line_synthetic", "diff_ewm_linha", "z_ewm_linha"]}, name="AUC no treino").to_frame()

## 2. Seleção (validação cruzada temporal + validação 2024-25)

In [ ]:
c = TV["classificacao"]
sel = pd.DataFrame({k: {"logloss CV": -v["score_cv_medio"], "parametros": v["melhores_parametros"]} for k, v in c["cv"].items()}).T
sel["logloss validação"] = [c["validacao"][k]["log_loss"] for k in sel.index]
sel["AUC validação"] = [c["validacao"][k]["auc"] for k in sel.index]
print("modelo final:", c["modelo_final"], "| calibração escolhida:", c["calibracao"]["metodo"])
print("Brier na 2a metade da validação por método:", c["calibracao"]["brier_segunda_metade"])
sel.sort_values("logloss validação")

A calibração foi escolhida **sem olhar o teste**: cada calibrador (Platt/sigmoid e isotônico) foi ajustado na primeira metade da validação e julgado na segunda. A regressão logística já sai calibrada, e nenhum calibrador melhorou o Brier, então o modelo final fica sem ajuste adicional.

## 3. Teste 2025-26

In [ ]:
cols = ["acuracia", "precisao", "revocacao", "f1", "auc", "brier", "brier_ingenuo", "brier_skill", "log_loss", "ece"]
a = pd.DataFrame(TF["classificacao"]["teste"]).T[cols]
b = pd.DataFrame(TF["classificacao"]["teste_sem_empates"]).T[cols]
pd.concat({"todas as linhas": a, "sem empates": b})

In [ ]:
print("regra sem treino (média exponencial acima da linha):", TF["classificacao"]["regra_exponencial_acima_da_linha"])
print("AUC do modelo, IC por jogador:", TF["classificacao"]["auc_ic"])
print("Brier modelo - ingênuo, IC por jogador:", TF["classificacao"]["brier_vs_ingenuo"])

### Ablação das features da classificação

In [ ]:
pd.DataFrame(TF["classificacao"]["ablacao"]).T[["acuracia", "auc", "brier", "brier_skill", "log_loss"]]

In [ ]:
display(Image("results/figures/f_curva_calibracao.png", width=520))
pd.DataFrame(TF["classificacao"]["curva_calibracao"])

## Leitura

- O classificador supera o estimador ingênuo (probabilidade constante): Brier menor com IC longe de zero e AUC ≈ 0,63, acima da regra simples "média exponencial acima da linha" (≈ 0,61).
- As probabilidades são calibradas: a curva acompanha a diagonal (quando o modelo diz 60%, o jogador supera a linha perto de 60% das vezes). O ECE baixo é necessário mas não suficiente, porque o estimador constante também o tem; a vantagem está na resolução (Brier e AUC).
- Os empates (3,3% do teste) não explicam o resultado: sem eles as métricas praticamente não mudam, e remover `linha_inteira` custa pouco.
- Sem as features relativas à linha (`so_pre_jogo`) a AUC cai de 0,630 para 0,602: elas acrescentam sinal, mas boa parte já vem das variáveis de volume, minutos e histórico. O sinal mistura tendência recente, assimetria e reversão ao patamar da temporada.
- O ganho é modesto, como previsto: a linha é a mediana do próprio jogador e a maior parte da variação de um jogo para outro não é previsível com informação pré-jogo.